## 1. 根据氨基酸序列表找到对应的核苷酸序列

In [1]:
import pandas as pd

Crick_H1N1 = pd.read_excel('../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../data/raw/data4model(Crick-H3N2).xlsx')

In [2]:
def read_fasta(file_path):
    sequence_names = []
    sequences = []
    current_sequence = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                # 如果当前序列不为空，保存当前序列
                if current_sequence:
                    sequences.append(''.join(current_sequence))
                    current_sequence = []
                # 保存序列名称
                sequence_names.append(line[1:])
            else:
                # 将序列数据添加到当前序列
                current_sequence.append(line)

        # 不要忘记保存最后一个序列
        if current_sequence:
            sequences.append(''.join(current_sequence))

    return sequence_names, sequences

def seq2DF(names, seqs):
    Meta = pd.DataFrame([s.split('|') for s in names])
    Meta['Sequence'] = seqs
    return Meta

In [3]:
sequence_names, sequences = read_fasta('../../data/processed/H1_DNA.fasta')
H1_DNA = seq2DF(sequence_names, sequences)
H1_DNA.columns = ['virusName','EPI_Isl_ID','Passage','lineage','collectionDate','EPI_ID','Sequence']
sequence_names, sequences = read_fasta('../../data/processed/H3_DNA.fasta')
H3_DNA = seq2DF(sequence_names, sequences)
H3_DNA.columns = ['virusName','EPI_Isl_ID','Passage','lineage','collectionDate','EPI_ID','Sequence']

In [4]:
sequence_names, sequences = read_fasta('../../data/processed/H1_AA.fasta')
H1_AA = seq2DF(sequence_names, sequences)
H1_AA = H1_AA.iloc[:,[0,1,3,6,14,16]].copy()
H1_AA.columns = ['virusName','EPI_Isl_ID','Passage','collectionDate','EPI_ID','Sequence']
sequence_names, sequences = read_fasta('../../data/processed/H3_AA.fasta')
H3_AA = seq2DF(sequence_names, sequences)
H3_AA = H3_AA.iloc[:,[0,1,3,6,14,17]].copy()
H3_AA.columns = ['virusName','EPI_Isl_ID','Passage','collectionDate','EPI_ID','Sequence']

In [5]:
H1N1_integrated = Crick_H1N1[['serumName','serumPassage', 'serumPassCat', 'virusName', 
                              'virusPassage','virusPassCat', 'serumIslID', 'virusIslID',
                              'serumHA', 'virusHA', 'HI_Dist']].copy()
H3N2_integrated = Crick_H3N2[['serumName','serumPassage', 'serumPassCat', 'virusName', 
                              'virusPassage','virusPassCat', 'serumIslID', 'virusIslID',
                              'serumHA', 'virusHA', 'HI_Dist']].copy()
H1N1_integrated['serumType'] = 'H1N1'
H3N2_integrated['serumType'] = 'H3N2'
H1N1_integrated.rename(columns={'serumHA':'serumHA_AA','virusHA':'virusHA_AA'},inplace=True)
H3N2_integrated.rename(columns={'serumHA':'serumHA_AA','virusHA':'virusHA_AA'},inplace=True)

In [8]:
filt1 = pd.merge(H1N1_integrated,H1_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['serumIslID','serumHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'serum_HA_EPI_ID'})
filt2 = pd.merge(filt1,H1_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['virusIslID','virusHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'virus_HA_EPI_ID'})

filt3 = pd.merge(filt2,H1_DNA[['EPI_ID','Sequence']],
                 left_on=['serum_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'serumHA_DNA'})
filt4 = pd.merge(filt3,H1_DNA[['EPI_ID','Sequence']],
                 left_on=['virus_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'virusHA_DNA'})
H1N1_final = filt4.copy()

filt1 = pd.merge(H3N2_integrated,H3_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['serumIslID','serumHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'serum_HA_EPI_ID'})
filt2 = pd.merge(filt1,H3_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['virusIslID','virusHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'virus_HA_EPI_ID'})

filt3 = pd.merge(filt2,H3_DNA[['EPI_ID','Sequence']],
                 left_on=['serum_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'serumHA_DNA'})
filt4 = pd.merge(filt3,H3_DNA[['EPI_ID','Sequence']],
                 left_on=['virus_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'virusHA_DNA'})
H3N2_final = filt4.copy()

In [19]:
Final_df = pd.concat([H1N1_final, H3N2_final], axis=0).reset_index(drop=True)

In [23]:
Final_df.to_csv('../../data/processed/0.1_Final_df.csv',index=False)

In [46]:
pd.merge(H3N2_integrated,H3N2_DNA_HA[['virusName','EPI_Isl_ID','Sequence','Passage']],
         left_on=['serumIslID','serumPassage'],
         right_on=['EPI_Isl_ID','Passage'],how='left')

,serumName,serumPassage,serumPassCat,virusName_x,virusPassage,virusPassCat,serumIslID,virusIslID,serumHA_AA,virusHA_AA,HI_Dist,serumType,virusName_y,EPI_Isl_ID,Sequence,Passage
0,A/CHRISTCHURCH/28/2003,Ex,EGG,A/WELLINGTON/1/2004,Ex,EGG,EPI_ISL_177752,EPI_ISL_127603,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,0.0,H3N2,NaN,NaN,NaN,NaN
1,A/CHRISTCHURCH/28/2003,Ex,EGG,A/TENNESSEE/6/2004,"M2,C1\1",CELL,EPI_ISL_177752,EPI_ISL_11815,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,QKLPGNDNSTATLCLGHHAVPNGTIVKTITNDQIEVTNATELVQSS...,5.0,H3N2,NaN,NaN,NaN,NaN
2,A/WELLINGTON/1/2004,Ex,EGG,A/CHRISTCHURCH/28/2003,Ex,EGG,EPI_ISL_127603,EPI_ISL_177752,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,4.0,H3N2,NaN,NaN,NaN,NaN
3,A/WELLINGTON/1/2004,Ex,EGG,A/TENNESSEE/6/2004,"M2,C1\1",CELL,EPI_ISL_127603,EPI_ISL_11815,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,QKLPGNDNSTATLCLGHHAVPNGTIVKTITNDQIEVTNATELVQSS...,5.0,H3N2,NaN,NaN,NaN,NaN
4,A/WELLINGTON/1/2004,Ex,EGG,A/SINGAPORE/37/2004,E4,EGG,EPI_ISL_127603,EPI_ISL_12068,MKTIIALSYILCLVFAQKLPGNDNSTATLCLGHHAVPNGTIVKTIT...,QKLPGNDNSTATLCLGHHAVPNGTIVKTITNDQIEVTNATELVQSS...,2.0,H3N2,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70898,A/ALBANIA/289813/2022,MDCK1/SIAT1,CELL,A/SLOVENIA/49/2024,MDCKx/SIAT1,CELL,EPI_ISL_18864899,EPI_ISL_18949962,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,2.0,H3N2,NaN,NaN,NaN,NaN
70899,A/ALBANIA/289813/2022,MDCK1/SIAT1,CELL,A/SLOVENIA/62/2024,SIATx/SIAT1,CELL,EPI_ISL_18864899,EPI_ISL_18949961,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,1.0,H3N2,NaN,NaN,NaN,NaN
70900,A/ALBANIA/289813/2022,MDCK1/SIAT1,CELL,A/SLOVENIA/119/2024,MDCKFx/SIAT1,CELL,EPI_ISL_18864899,EPI_ISL_18949957,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,1.0,H3N2,NaN,NaN,NaN,NaN
70901,A/ALBANIA/289813/2022,MDCK1/SIAT1,CELL,A/SLOVENIA/145/2024,MDCKFx/SIAT1,CELL,EPI_ISL_18864899,EPI_ISL_18949960,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...,MKAIIALSNILCLVFAQKIPGNDNSMATLCLGHHAVSNGTIVKTIT...,1.0,H3N2,NaN,NaN,NaN,NaN
